# Statistical analysis of SFC

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from statsmodels.formula.api import ols, glm
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import os
import plotly.express as px
import plotly.graph_objects as go
import urllib.request
import plotly.graph_objs as go
from statsmodels.genmod import families
from statsmodels.genmod.families import links
from scipy.stats import linregress
from scipy.stats import pearsonr

In [ ]:
# Get the current working directory
wd = os.getcwd()
wd = (wd[:-10]+'/')

#### ADJUST#####
analysis = 'whole' 
files = 'fbc_fc'
folder = '216'
path = wd
lebs = '/data/atlas_labels.csv'

#############

#load coupling results for both groups and the labels for the regions
hc = pd.read_csv(path+folder+'/results/'+files+'/'+analysis+'/hc_'+files+'.csv')
bd = pd.read_csv(path+folder+'/results/'+files+'/'+analysis+'/bd_'+files+'.csv')
leb = pd.read_csv(path+folder+lebs,header=None)
leb.rename(columns={leb.columns[0]: 'regions'}, inplace=True)
leb['regions'] = leb['regions'].str.replace(' ', '', regex=True)
#load covariates
hc_cov = pd.read_csv(path+folder+'/data/HC_covar.csv')
bd_cov = pd.read_csv(path+folder+'/data/BD_covar.csv')

# Match r2 values shape to covariates shape and set region labels as column names
hc = hc.T 
bd = bd.T
hc.columns = leb['regions'].tolist()
bd.columns = leb['regions'].tolist()

#assign group 
hc_cov['gr'] = 0
bd_cov['gr'] = 1

# convert categorical covariates to dummies and set as categorical
hc_cov['sex'] = hc_cov['sex'].map({'Male': 0, 'Female': 1}).astype('category')
hc_cov['centre'] = hc_cov['centre'].map({'Cheadle (imaging)': 0, 'Newcastle (imaging)': 1,'Reading (imaging)':2,
                                         'Bristol (imaging)':3}).astype('category')

bd_cov['sex'] = bd_cov['sex'].map({'Male': 0, 'Female': 1}).astype('category')
bd_cov['centre'] = bd_cov['centre'].map({'Cheadle (imaging)': 0, 'Newcastle (imaging)': 1, 'Reading (imaging)':2, 
                                         'Bristol (imaging)':3}).astype('category')



hc_cov['gr'] = hc_cov['gr'].astype('category')
bd_cov['gr'] = bd_cov['gr'].astype('category')

# Match index type for concatenation and concatenate data with covariates
hc.index = hc.index.astype('int64')
bd.index = bd.index.astype('int64')

hc_cov.index = hc_cov.index.astype('int64')
bd_cov.index = bd_cov.index.astype('int64')

hc_f= pd.concat([hc_cov, hc], axis=1)
bd_f= pd.concat([bd_cov, bd], axis=1)
hc_f.set_index('eid', inplace=True)
bd_f.set_index('eid', inplace=True) # now for each group hc_f/bd_f is the full data of coupling and covariates


# concatinate both groups for full datasets to be used in GLM
data = pd.concat([bd_f, hc_f], axis=0)

# make sure all categorical covariates are type categorical
data['sex'] = data['sex'].astype('category')
data['centre'] = data['centre'].astype('category')
data['gr'] = data['gr'].astype('category')

raw_data = data.iloc[:,7:]# This is the coupling for both groups without covariates

#### Demographics

In [ ]:
# count the female and male in each group and age data
counts = hc_cov['sex'].value_counts()
print('HC females', counts[1], 'males', counts[0])
female_per = (counts[1]/(counts[1]+counts[0]))*100
print('percent of females: ', round(female_per), '\n')
      
counts = bd_cov['sex'].value_counts()
print('BD females', counts[1], 'males', counts[0],'\n')
female_per = counts[1]/(counts[1]+counts[0])*100
print('percent of females: ', round(female_per), '\n')

print('Mean age: HC' , round(hc_cov['age'].mean(),1), 'BD' , round(bd_cov['age'].mean(),1))
print('SD age: HC' , round(hc_cov['age'].std(),1), 'BD' , round(bd_cov['age'].std(),1))
print('Min age: HC' , hc_cov['age'].min(), 'BD' , bd_cov['age'].min())
print('Max age: HC' , hc_cov['age'].max(), 'BD' , bd_cov['age'].max())

#### Individual level SFC

In [ ]:
raw_data.describe().loc[['max'],:].round(4).sort_values(by='max', axis=1, ascending=False).iloc[:,:20]

for participants (n = 2) that had a region with empty values across all timepoints in their timeseries - need to replace with mean R2 of the region

In [ ]:
clean_data = raw_data.copy()

def replace_ones_with_mean(data):
    if isinstance(data, pd.DataFrame):
        for column in data.columns:
            if 1 in data[column].values:  # Check if 1 is in the column
                column_mean = data[column][data[column] != 1].mean()  # Calculate mean excluding 1
                data[column] = data[column].replace(1, column_mean)   # Replace 1 with the mean
                print(f"Column '{column}' had 1 replaced with mean")
    
    elif isinstance(data, pd.Series):
        if 1 in data.values:  # Check if 1 is in the Series
            series_mean = data[data != 1].mean()  # Calculate mean excluding 1
            data = data.replace(1, series_mean)   # Replace 1 with the mean
    return data  # Return the modified Series

clean_data=replace_ones_with_mean(clean_data)

In [ ]:
print("\nHighest coupled by individual values:\n",
      clean_data.describe().loc[['max'],:].round(4).sort_values(by='max', axis=1, ascending=False).iloc[:,:8],"\n")

print("\nLowest coupled by individual values:\n",
      clean_data.describe().loc[['min'],:].round(4).sort_values(by='min', axis=1, ascending=True).iloc[:,:8])

total_mean = clean_data.values.mean()
total_median = np.median(clean_data.values)
print("\nmean coupled by individual values:\n",
      total_mean)

print("\nmedian coupled by individual values:\n",
      total_median)

In [ ]:
# Create indexing per hemisphere for the plots based on the length of the data
if leb.shape[0] < 220:
    left = slice(None, 100)
    right = slice(100, 200)
    subr = slice(200, 208)
    subl = slice(208, None)
    cortical = '200'
    subcortical = '1'

else:
    left = slice(None, 250)
    right = slice(250, 500)
    subr = slice(500, 527)
    subl = slice(527, None)
    cortical = '500'
    subcortical = '4'        

In [ ]:
# Fetch the Schaefer parcellation data from the URL
url = 'https://raw.githubusercontent.com/ThomasYeoLab/CBIG/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Centroid_coordinates/Schaefer2018_'+cortical+'Parcels_7Networks_order_FSLMNI152_1mm.Centroid_RAS.csv'
response = urllib.request.urlopen(url)
sch = np.genfromtxt(response, delimiter=',')
sch = sch[1:]
sch = sch[:, 2:]

# Load the Tian subcortex data from the file (unless analysis scope was cortical)

file_path = (wd+'/atlases/Tian_Subcortex_S'+subcortical+'_3T_COG.txt')
tia = np.loadtxt(file_path)
coords = np.vstack((sch, tia))  
coords=coords.astype(int)

#store labels with coords for later plotting
label_coord = {}
for l ,c in zip(leb['regions'], coords):
    label_coord[l] = c

#### Regional SFC means (averaged)

In [ ]:
# calculate the mean coupling for each region across participants
hc_mean = hc.mean()
bd_mean = bd.mean()
data_mean = clean_data.mean()
data_sd = clean_data.std()
data_min = clean_data.min()
data_max = clean_data.max()

# get the descriptives of the regional means
data_stats = pd.DataFrame({'Mean': data_mean.round(3), 'SD': data_sd.round(2), 'min': data_min.round(2), 'max': data_max.round(2)})
data_mean_max = data_stats.sort_values(by='Mean', ascending=False)
data_mean_min = data_stats.sort_values(by='Mean', ascending=True)


In [ ]:
print('Highest coupled regions averaged across participants:')
max_mean = data_mean_max.head(8).copy()
coord_max = [label_coord[idx] for idx in max_mean.index if idx in label_coord]
max_mean['Coordinates'] = coord_max

print(max_mean)

In [ ]:
print('Lowest coupled regions averaged across participants:')
min_mean = data_mean_min.head(8).copy()
coord_min = [label_coord[idx] for idx in min_mean.index if idx in label_coord]
min_mean['Coordinates'] = coord_min

print(min_mean)

Plot the regions for each hemisphere

In [ ]:
# initiate colors
c1='lightpink'
c2='teal'

# loop over the networks to assign them color for the bar plot
for ix, s in enumerate(leb['regions']):    
    if 'Vis' in s:
        leb.loc[ix,'networks']= 'Visual'
        leb.loc[ix,'colors']= c1
    elif 'Som' in s:
        leb.loc[ix,'networks']= 'SMN'
        leb.loc[ix,'colors']= c2
    elif 'Dor' in s:
        leb.loc[ix,'networks']= 'DAN'
        leb.loc[ix,'colors']= c1
    elif 'Sal' in s:
        leb.loc[ix,'networks']= 'VAN'
        leb.loc[ix,'colors']= c2       
    elif 'Lim' in s:
        leb.loc[ix,'networks']= 'Limbic'
        leb.loc[ix,'colors']= c1
    elif 'Cont' in s:
        leb.loc[ix,'networks']= 'ECN'
        leb.loc[ix,'colors']= c2
    elif 'Def' in s:
        leb.loc[ix,'networks']= 'DMN'
        leb.loc[ix,'colors']=c1 
    else:
        leb.loc[ix,'networks']= 'Subcortex'
        leb.loc[ix,'colors']= c2

In [ ]:
leb['plot'] = leb['regions'].str.replace('L_', '', regex=True)
leb['plot'] = leb['plot'].str.replace('R_', '', regex=True)
leb['plot'] = leb['plot'].str.replace('_', ' ', regex=True)
leb['plot'] = leb['plot'].str.title()

#set y range so all plots will be on the same y limits
y_min = data_mean.values.min()
y_max = data_mean.values.max()
y_margin = (y_max - y_min) * 0.05
y_range = [y_min - y_margin, y_max + y_margin]

####### LEFT HEMISPHERE ####
x_inner = leb['plot'].iloc[left].tolist()+ leb['plot'].iloc[subl].tolist()
x_outer = leb['networks'].iloc[left].tolist() +leb['networks'].iloc[subl].tolist()
cc = leb['colors'].iloc[left].tolist() +leb['colors'].iloc[subl].tolist()
x = [x_outer,x_inner]
bar_trace = go.Bar(
    x=x,  
    y=data_mean.iloc[left].tolist()+data_mean.iloc[subl].tolist(),
    orientation='v',
    name='Mean',
    marker_color=cc)

layout = go.Layout(
    title='Regional SC-FC Mean - Left Hemisphere',
    xaxis=dict(
        title='Region',
        type='multicategory',
        tickangle=90,
        automargin=True,
        mirror=True,
        ticks='outside',
        showline=False,
        linecolor='dimgray',
        linewidth=0.2,
        tickfont=dict(size=9),
    ),
    yaxis=dict(title='Mean', range=y_range, 
        showgrid=True,
        mirror=True,
        ticks='outside',
        showline=False,
        linecolor='dimgray',
        dtick=0.05,
        gridcolor='grey'               
),  
    width=1100,
    height=800,    
    plot_bgcolor='white',  # Set plot background color to white
    paper_bgcolor='white'
)
fig = go.Figure(data=[bar_trace], layout=layout)
fig.show()


####### RIGHT HEMISPHERE ####
x_inner = leb['plot'].iloc[right].tolist()+ leb['plot'].iloc[subr].tolist()
x_outer = leb['networks'].iloc[right].tolist() +leb['networks'].iloc[subr].tolist()
cc = leb['colors'].iloc[right].tolist() +leb['colors'].iloc[subr].tolist()
x = [x_outer,x_inner]
bar_trace = go.Bar(
    x=x,  
    y=data_mean.iloc[right].tolist()+data_mean.iloc[subr].tolist(),
    orientation='v',
    name='Mean',
    marker_color=cc)

layout = go.Layout(
    title='Regional SC-FC Mean - Right Hemisphere',
    xaxis=dict(
        title='Region',
        type='multicategory',
        tickangle=90,
        automargin=True,
        mirror=True,
        ticks='outside',
        showline=False,
        linecolor='dimgray',
        linewidth=0.2,
        tickfont=dict(size=9),
    ),
    yaxis=dict(title='Mean', range=y_range, 
        showgrid=True,
        mirror=True,
        ticks='outside',
        showline=False,
        linecolor='dimgray',
        dtick=0.05,
        gridcolor='grey'               
),  
    width=1100,
    height=800,    
    plot_bgcolor='white',  # Set plot background color to white
    paper_bgcolor='white'
)
fig = go.Figure(data=[bar_trace], layout=layout)
fig.show()

#### Means by network

In [ ]:
# lists to store the indices of each network
vis = [] 
smn = []
dan = []
van = []
lim = []
dmn = []
ecn = []
sub = []

# loop to extract the ondices of each network
for ix, s in enumerate(leb['regions']):
    if 'Vis' in s:
        vis.append(ix)
    elif 'Som' in s:
        smn.append(ix)
    elif 'Dor' in s:
        dan.append(ix)
    elif 'Sal' in s:
        van.append(ix)
    elif 'Lim' in s:
        lim.append(ix)
    elif 'Cont' in s:
        ecn.append(ix)
    elif 'Def' in s:
        dmn.append(ix)
    else:
        sub.append(ix)

# loop to print the means of each network in both groups and in total
groups = [hc_mean, bd_mean, data_mean]
group_names = ['HC', 'BD', 'Both']
means = []
stds = []

for group, name in zip(groups, group_names):
    print(f"{name}:")
    visual = group.iloc[vis]
    somatomotor = group.iloc[smn]
    dor_at = group.iloc[dan]
    sal = group.iloc[van]
    limbic = group.iloc[lim]
    ecnn = group.iloc[ecn]
    dmnn = group.iloc[dmn]
    subb = group.iloc[sub]
    
    print(f"Vis mean: {visual.mean():.3f}, std: {visual.std():.3f}, max: {visual.max():.3f}, min: {visual.min():.3f}")
    print(f"SMN mean: {somatomotor.mean():.3f}, std: {somatomotor.std():.3f}, max: {somatomotor.max():.3f}, min: {somatomotor.min():.3f}")
    print(f"DAN mean: {dor_at.mean():.3f}, std: {dor_at.std():.3f}, max: {dor_at.max():.3f}, min: {dor_at.min():.3f}")
    print(f"VAN mean: {sal.mean():.3f}, std: {sal.std():.3f}, max: {sal.max():.3f}, min: {sal.min():.3f}")
    print(f"Lim mean: {limbic.mean():.3f}, std: {limbic.std():.3f}, max: {limbic.max():.3f}, min: {limbic.min():.3f}")
    print(f"CEN mean: {ecnn.mean():.3f}, std: {ecnn.std():.3f}, max: {ecnn.max():.3f}, min: {ecnn.min():.3f}")
    print(f"DMN mean: {dmnn.mean():.3f}, std: {dmnn.std():.3f}, max: {dmnn.max():.3f}, min: {dmnn.min():.3f}")
    print(f"Sub mean: {subb.mean():.3f}, std: {subb.std():.3f}, max: {subb.max():.3f}, min: {subb.min():.3f} \n")

In [ ]:
# Create a dictionary of network values for the HC group
hc_network_values = {
    'Vis': hc_mean.iloc[vis].tolist(),
    'SMN': hc_mean.iloc[smn].tolist(),
    'DAN': hc_mean.iloc[dan].tolist(),
    'VAN': hc_mean.iloc[van].tolist(),
    'Lim': hc_mean.iloc[lim].tolist(),
    'CEN': hc_mean.iloc[ecn].tolist(),
    'DMN': hc_mean.iloc[dmn].tolist(),
    'Sub': hc_mean.iloc[sub].tolist()
}

# Create a DataFrame from the dictionary for the HC group
hc_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in hc_network_values.items()]))

print("\nHC Group DataFrame (with individual values):")
hc_df.describe().sort_values(by='50%', axis=1, ascending=False)

In [ ]:
# Create a dictionary of network values for the BD group
bd_network_values = {
    'Vis': bd_mean.iloc[vis].tolist(),
    'SMN': bd_mean.iloc[smn].tolist(),
    'DAN': bd_mean.iloc[dan].tolist(),
    'VAN': bd_mean.iloc[van].tolist(),
    'Lim': bd_mean.iloc[lim].tolist(),
    'CEN': bd_mean.iloc[ecn].tolist(),
    'DMN': bd_mean.iloc[dmn].tolist(),
    'Sub': bd_mean.iloc[sub].tolist()
}

# Create a DataFrame from the dictionary for the BD group
bd_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in bd_network_values.items()]))

print("\nBD Group DataFrame (with individual values):")
bd_df.describe().sort_values(by='50%', axis=1, ascending=False)

In [ ]:
print("Creating DataFrames with actual values for the all")

# Create a dictionary of network values for all
data_network_values = {
    'Vis': data_mean.iloc[vis].tolist(),
    'DAN': data_mean.iloc[dan].tolist(),
    'SMN': data_mean.iloc[smn].tolist(),
    'DMN': data_mean.iloc[dmn].tolist(),
    'CEN': data_mean.iloc[ecn].tolist(),
    'VAN': data_mean.iloc[van].tolist(),
    'Lim': data_mean.iloc[lim].tolist(),
    'Sub': data_mean.iloc[sub].tolist()
}

# Create a DataFrame from the dictionary for all
data_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in data_network_values.items()]))

print("\nData Group DataFrame (with individual values):")
data_df.describe().sort_values(by='50%', axis=1, ascending=False)


In [ ]:
# box plot for the networks
df_melted = data_df.melt(var_name='Network', value_name='Mean R²')
fig = go.Figure()
networks = df_melted['Network'].unique()
colors = px.colors.qualitative.Plotly

for i, network in enumerate(networks):
    fig.add_trace(go.Violin(
        y=df_melted[df_melted['Network'] == network]['Mean R²'],
        name=network,
        box=dict(line_color='black', line_width=1.5),  
        meanline=dict(visible=True, color='black', width=1.5),
        fillcolor=colors[i % len(colors)],
        line_color='black',
        opacity=0.7,
        line=dict(color='black', width=1),
        showlegend=False,
        x0=network
    ))

fig.update_layout(
    title_text="",
    xaxis_title="",
    yaxis_title="",
    width=800,
    height=500,
    template="simple_white",
    margin=dict(l=50, r=20, t=20, b=50),
    xaxis=dict(
        tickfont=dict(size=14)  # Increase network label size
    ),
    yaxis=dict(
        tickfont=dict(size=14)  # Increase network label size
    ),
)

fig.show()
#fig.write_image(wd+f'/PLOTS/networks_{analysis}_{files}_{folder}.svg')

#### Group comparison

Perform a GLM with a Gamma family function and log link function (R2 vdisribution is pretty right sqewed):

In [ ]:
model_summaries = []  # store model summaries
group_p_values = []  # store p-values for 'group'
group_z_values = []  # store F-values for 'group'
group_coefficients = []  # store coefficients for 'group'
group_CI25 =[]
group_CI9 = []


# Perform GLM for each outcome variable column separately
for col in data.columns[7:]:  # outcome variable columns start from index 7
    y = data[col].copy()  # store the outcome variable
    ys = replace_ones_with_mean(y)  # for error measurements (missing rsFC in two participants)  

    #for Gamma with log, values should be positive
    min_value = ys.min() 
    shift_amount = abs(min_value) + 1e-10
    ys = ys + shift_amount #shift all data points so all are positive
    
    temp_data = pd.concat([ys, data[['gr', 'age', 'sex']]], axis=1) #concat each time a new y with the IVs
    temp_data.columns = [col, 'gr', 'age', 'sex'] # name the columns

    # Perform GLM
    formula = f'{col} ~ C(gr) + age + C(sex)'  # GLM formula
    model = sm.GLM.from_formula(formula, data=temp_data, family=families.Gamma(link=links.Log())).fit()
    
    # Store GLM summary
    summary = model.summary2().tables[1]  # extract the summary table
    model_summaries.append(summary)  # add the summary table to the list
    
    # Extract p-value and z-value for 'group'
    group_p_values.append(summary.loc['C(gr)[T.1]', 'P>|z|'])
    group_z_values.append(summary.loc['C(gr)[T.1]', 'z'])
    group_coefficients.append(summary.loc['C(gr)[T.1]', 'Coef.'])
    group_CI25.append(summary.loc['C(gr)[T.1]','[0.025'])
    group_CI9.append(summary.loc['C(gr)[T.1]', '0.975]'])


# Apply correction to the p-values for 'group'
_, gr_corrected_p_values, _, _ = multipletests(group_p_values, method='fdr_bh')
gr_corrected_p_values_df = pd.DataFrame(gr_corrected_p_values, index=data.columns[7:], columns=['adj_p-value'])
gr_z_values_df = pd.DataFrame(group_z_values, index=data.columns[7:], columns=['z-value'])
gr_p_values_df = pd.DataFrame(group_p_values, index=data.columns[7:], columns=['p-value'])
gr_coefficients_df = pd.DataFrame(group_coefficients, index=data.columns[7:], columns=['b'])
gr_CI25_df = pd.DataFrame(group_CI25, index=data.columns[7:], columns=['CI 0.025'])
gr_CI9_df = pd.DataFrame(group_CI9, index=data.columns[7:], columns=['CI 0.975'])

# Concatenate F-values, FDR corrected p-values, raw p-values, and coefficients into a single DataFrame
group_result_df = pd.concat([gr_z_values_df, gr_corrected_p_values_df, gr_p_values_df, gr_coefficients_df,gr_CI25_df,gr_CI9_df], axis=1)
main_top_hits = group_result_df[group_result_df['adj_p-value'] <= 0.05]
group_main_top_hits = main_top_hits.sort_values(by='adj_p-value', ascending=True)
coord_hits_all = [label_coord[idx] for idx in group_result_df.index if idx in label_coord]
group_result_df['Coordinates']=coord_hits_all

# Print summaries only for the main top hits
for idx, col in enumerate(data.columns[7:]):
    if col in group_main_top_hits.index:
        print(f"GLM Summary for Outcome Variable {col} adjusted for age and sex:")
        print(model_summaries[idx])

# View the final results
coord_hits = [label_coord[idx] for idx in group_main_top_hits.index if idx in label_coord]

group_main_top_hits['Coordinates']=coord_hits
group_main_top_hits

In [ ]:
print('adjusted for age and sex')
group_result_df.sort_values(by='p-value', ascending=True).head(5)

with full covariates:

In [ ]:
model_summaries = []
group_p_values = []  
group_z_values = [] 
group_coefficients = []  
group_CI25 =[]
group_CI9 = []


for col in data.columns[7:]:  
    y = data[col].copy()  
    ys = replace_ones_with_mean(y)
    min_value = ys.min() 
    shift_amount = abs(min_value) + 1e-10
    ys = ys + shift_amount 
    
    temp_data = pd.concat([ys, data[['gr', 'age', 'sex', 'centre', 'd_motion', 'f_motion', 'icv']]], axis=1)
    temp_data.columns = [col, 'gr', 'age', 'sex', 'centre', 'd_motion', 'f_motion', 'icv']

    # Perform GLM
    formula = f'{col} ~ C(gr) + age + C(sex) + C(centre) + d_motion + f_motion + icv'
    model = sm.GLM.from_formula(formula, data=temp_data, family=families.Gamma(link=links.Log())).fit()
    
    summary = model.summary2().tables[1]
    model_summaries.append(summary) 
    
    group_p_values.append(summary.loc['C(gr)[T.1]', 'P>|z|'])
    group_z_values.append(summary.loc['C(gr)[T.1]', 'z'])
    group_coefficients.append(summary.loc['C(gr)[T.1]', 'Coef.'])
    group_CI25.append(summary.loc['C(gr)[T.1]','[0.025'])
    group_CI9.append(summary.loc['C(gr)[T.1]', '0.975]'])


_, gr_corrected_p_values, _, _ = multipletests(group_p_values, method='fdr_bh')
gr_corrected_p_values_df = pd.DataFrame(gr_corrected_p_values, index=data.columns[7:], columns=['adj_p-value'])
gr_z_values_df = pd.DataFrame(group_z_values, index=data.columns[7:], columns=['z-value'])
gr_p_values_df = pd.DataFrame(group_p_values, index=data.columns[7:], columns=['p-value'])
gr_coefficients_df = pd.DataFrame(group_coefficients, index=data.columns[7:], columns=['b'])
gr_CI25_df = pd.DataFrame(group_CI25, index=data.columns[7:], columns=['CI 0.025'])
gr_CI9_df = pd.DataFrame(group_CI9, index=data.columns[7:], columns=['CI 0.975'])

group_result_df = pd.concat([gr_z_values_df, gr_corrected_p_values_df, gr_p_values_df, gr_coefficients_df,gr_CI25_df,gr_CI9_df], axis=1)
main_top_hits2 = group_result_df[group_result_df['adj_p-value'] <= 0.05]
group_main_top_hits2 = main_top_hits2.sort_values(by='adj_p-value', ascending=True)
coord_hits_all = [label_coord[idx] for idx in group_result_df.index if idx in label_coord]
group_result_df['Coordinates']=coord_hits_all

for idx, col in enumerate(data.columns[7:]):
    if col in group_main_top_hits2.index:
        print(f"GLM Summary for Outcome Variable {col} asjuested for age, sex, icv, motion:")
        print(model_summaries[idx])

coord_hits = [label_coord[idx] for idx in group_main_top_hits2.index if idx in label_coord]

group_main_top_hits2['Coordinates']=coord_hits
group_main_top_hits2

In [ ]:
## values to compare top regions across the two parcellations
cols = ['adj_p-value','p-value','b','CI 0.025','CI 0.975','Coordinates']

rows_554 = [
    'L_Limbic_TempPole_5',
    'R_DorsAttn_FEF_2',
    'L_SomMot_32',
    'L_SalVentAttn_ParOper_5',
    'R_Cont_PFCl_5_frontal_pole',
    'R_SomMot_17'
]

rows_216 = [
    'L_Limbic_TempPole_3_temporal_pole',
    'R_DorsAttn_FEF_2_superior_frontal_gyrus',
    'L_SomMot_15_precentral_gyrus',
    'L_SalVentAttn_ParOper_3_anterior_supramarginal_gyrus',
    'R_Cont_PFCl_2_frontal_pole',
    'R_SomMot_7_postcentral_gyrus'
]


if folder == '554':
    print(group_result_df.loc[rows_554, cols])
elif folder == '216':
    print(group_result_df.loc[rows_216, cols])


plot significant results

In [ ]:
# check distribution of the top hits

# Loop through the top hit regions
for col in group_main_top_hits.index:
    if col in data.columns:
        # Convert to numeric just in case
        data[col] = pd.to_numeric(data[col], errors='coerce')

        # Drop NaNs
        data_clean = data[col].dropna()

        # Plot
        if not data_clean.empty:
            fig = px.histogram(
                data_clean,
                nbins=30,
                title=f'Distribution of {col} (Top Hit)',
                labels={'value': col}
            )
            fig.update_layout(
                xaxis_title=col,
                yaxis_title='Frequency'
            )
            fig.show()
        else:
            print(f"{col} is empty or non-numeric after conversion.")
    else:
        print(f"{col} not found in data columns.")


In [ ]:
# box plot for the top hits

for region, p in group_main_top_hits2.iterrows():
    hc_gp = hc[region]
    bd_gp = bd[region]
    
    hc_gp = replace_ones_with_mean(hc_gp)
    bd_gp = replace_ones_with_mean(bd_gp)

    fig = go.Figure()

    # Add Violin plots for Control Group and Bipolar Group
    fig.add_trace(go.Violin(
        y=hc_gp, name='Healthy controls', 
        marker_color='#43A17A', 
        box=dict(line_color='black', line_width=2),
        meanline=dict(visible=True, color='black', width=1.5),
        showlegend=False,
        points=False,
        line=dict(color='#43A17A', width=0.5)
    ))
    fig.add_trace(go.Violin(
        y=bd_gp, name='Bipolar disorder', 
        marker_color='#A14362', 
        box=dict(line_color='black', line_width=2),
        meanline=dict(visible=True, color='black', width=1.5),
        showlegend=False,
        points=False,
        line=dict(color='#A14362', width=0.5)
    ))

    fig.update_layout(
        #title="",
        yaxis_title=dict(
            text='R²',
            font=dict(size=14)
        ),
        width=500,
        height=600,
        template="simple_white",
        margin=dict(l=50, r=20, t=20, b=50),
        xaxis=dict(mirror=False, tickfont=dict(size=14)),
        yaxis=dict(
            dtick=0.05,
            mirror=False,
            ticks='outside',
            tickfont=dict(size=12)
        ),
        annotations=[
            dict(
                xref='paper', yref='paper',
                x=0.5, y=0.9,
                text=f"p-value={p['adj_p-value']:.3f}",
                showarrow=False,
                font=dict(size=14, color='black')
            )
        ],
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig.show()
   # fig.write_image(wd+f'/PLOTS/group_diff_{region}_{analysis}_{files}_{folder}.svg')

#### Strength & Degree

In [ ]:
strength_degree_path = path + folder + '/results/strength_degree/'

### Structural Degree and Strength ###
hc_degree_sc = pd.read_csv(strength_degree_path + 'hc_degree_fbc.csv')
bd_degree_sc = pd.read_csv(strength_degree_path + 'bd_degree_fbc.csv')
hc_strength_sc = pd.read_csv(strength_degree_path + 'hc_strength_fbc.csv')
bd_strength_sc = pd.read_csv(strength_degree_path + 'bd_strength_fbc.csv')

### Functional Degree and Strength ###
#hc_degree_fc = pd.read_csv(strength_degree_path + 'hc_degree_fc.csv')
#bd_degree_fc = pd.read_csv(strength_degree_path + 'bd_degree_fc.csv')
hc_strength_fc = pd.read_csv(strength_degree_path + 'hc_strength_fc.csv')
bd_strength_fc = pd.read_csv(strength_degree_path + 'bd_strength_fc.csv')

### Functional (partial correlation) Degree and Strength ###
#hc_degree_pfc = pd.read_csv(strength_degree_path + 'hc_degree_pfc.csv')
#bd_degree_pfc = pd.read_csv(strength_degree_path + 'bd_degree_pfc.csv')
hc_strength_pfc = pd.read_csv(strength_degree_path + 'hc_strength_pfc.csv')
bd_strength_pfc = pd.read_csv(strength_degree_path + 'bd_strength_pfc.csv')

# Set column names to region labels
region_names = leb['regions'].tolist()
for df in [hc_degree_sc, bd_degree_sc, 
           hc_strength_sc, bd_strength_sc, 
          # hc_degree_fc, bd_degree_fc,
           hc_strength_fc, bd_strength_fc, 
         #  hc_degree_pfc, bd_degree_pfc,
           hc_strength_pfc, bd_strength_pfc]:
    
    df.columns = region_names

# Set index to int (for matching IDs if needed)
for df in [hc_degree_sc, bd_degree_sc, 
           hc_strength_sc, bd_strength_sc, 
          # hc_degree_fc, bd_degree_fc,
           hc_strength_fc, bd_strength_fc, 
          # hc_degree_pfc, bd_degree_pfc,
           hc_strength_pfc, bd_strength_pfc]:
    df.index = df.index.astype('int64')

# Concatenate without covariates
degree_sc = pd.concat([hc_degree_sc, bd_degree_sc], axis=0)
strength_sc = pd.concat([hc_strength_sc, bd_strength_sc], axis=0)
#degree_fc = pd.concat([hc_degree_fc, bd_degree_fc], axis=0)
strength_fc = pd.concat([hc_strength_fc, bd_strength_fc], axis=0)
#degree_pfc = pd.concat([hc_degree_pfc, bd_degree_pfc], axis=0)
strength_pfc = pd.concat([hc_strength_pfc, bd_strength_pfc], axis=0)

# Compute mean across subjects (rows) for each region (column)
sc_degree_mean = degree_sc.mean()
sc_strength_mean = strength_sc.mean()
#fc_degree_mean = degree_fc.mean()
fc_strength_mean = strength_fc.mean()
#pfc_degree_mean = degree_pfc.mean()
pfc_strength_mean = strength_pfc.mean()


In [ ]:
# plot corr overall between SFC and SC-FC strength and degree

def make_single_plot(datas, data_mean, x_title, y_title, title_text=" "):
    slope, intercept, r_value, p_value, std_err = linregress(datas, data_mean)
    regression_line = slope * datas + intercept

    fig = go.Figure()

    # Scatter plot
    fig.add_trace(go.Scatter(
        x=datas, y=data_mean,
        mode='markers',
        marker=dict(color='dimgrey'),
        showlegend=False
    ))

    # Regression line
    fig.add_trace(go.Scatter(
        x=datas, y=regression_line,
        mode='lines',
        line=dict(color='#570000'),
        name='Regression line',
        showlegend=False
    ))

    # Annotation
    fig.add_annotation(
        x=max(datas),
        y=max(data_mean),
        text = f"<i>R</i><sup>2</sup> = {r_value**2:.2f}<br><i>p</i> = {p_value:.3f}",
        showarrow=False
    )

    # Axis labels
    fig.update_xaxes(title_text=x_title)
    fig.update_yaxes(title_text=y_title)

    # Layout
    fig.update_layout(
        font=dict(family="sans-serif", size=12, color="black"),
        height=300,
        width=450,
        title_text=title_text,
        margin=dict(l=60, r=40, t=80, b=40),
        plot_bgcolor='rgb(255, 234, 224)',
        paper_bgcolor='white'
    )
    return fig


# Choose data sets depending on condition
if files == 'fbc_fc':
    data_sets = [
        (sc_degree_mean, 'Structural Degree', 'R²'),
        (sc_strength_mean, 'Structural Strength', 'R²'),
        (fc_strength_mean, 'Functional Strength', 'R²')
    ]
else: # would bd pfc
    data_sets = [
        (sc_degree_mean, 'Structural Degree', 'R²'),
        (sc_strength_mean, 'Structural Strength', 'R²'),
        (pfc_strength_mean, 'Functional Strength', 'R²')
    ]

# Generate separate figures
fig1 = make_single_plot(data_sets[0][0], data_mean, data_sets[0][1], data_sets[0][2], title_text=" ")
fig2 = make_single_plot(data_sets[1][0], data_mean, data_sets[1][1], data_sets[1][2], title_text=" ")
fig3 = make_single_plot(data_sets[2][0], data_mean, data_sets[2][1], data_sets[2][2], title_text=" ")

# Show them one by one
fig1.show()
fig2.show()
fig3.show()

#fig1.write_image(f"{path}PLOTS/s_degree_{files}_{analysis}_{folder}.svg")
#fig2.write_image(f"{path}PLOTS/s_strength_{files}_{analysis}_{folder}.svg")
#fig3.write_image(f"{path}PLOTS/f_strength_{files}_{analysis}_{folder}.svg")


for specific region:

In [ ]:


for region, p in main_top_hits2.iterrows():
    x = degree_sc.loc[:, region]
    y = clean_data.loc[:, region]
    correlation, p_value = pearsonr(x, y)
    print(f'The correlation in {region} and structural degree is r = {correlation:.3f}, p = {p_value:.3g}')

In [ ]:
for region, p in main_top_hits2.iterrows():
    x = strength_sc.loc[:, region]
    y = clean_data.loc[:, region]
    correlation, p_value = pearsonr(x, y)
    print(f'The correlation in {region} and structural strength is r = {correlation:.3f}, p = {p_value:.3g}')

In [ ]:
for region, p in main_top_hits2.iterrows():
    x = strength_fc.loc[:, region]
    y = clean_data.loc[:, region]
    correlation, p_value = pearsonr(x, y)
    print(f'The correlation in {region} and functional strength is r = {correlation:.3f}, p = {p_value:.3g}')